In [4]:
import random
import hashlib
import binascii

In [6]:
wordlist = open('wordlists/english.txt').read().split('\n')

In [14]:
def number_to_words(number, wordlist):
    binstr = '{0:0128b}'.format(number)
    print(f'binstr: {binstr}')
    assert len(binstr) == 128
    hexstr = "{0:0>4X}".format(int(binstr,2)).zfill(32)
    print(f'hexstr: {hexstr}')
    ascbin = binascii.a2b_hex(hexstr)
    print(f'ascstr: {ascbin}')
    sha256 = hashlib.sha256(ascbin).hexdigest()
    print(f'sha256: {sha256}')  
    print(f'sha256[0]: {sha256[0]}')  
    print(f'int(sha256[0],16): {int(sha256[0],16)}')  
    print(f'bin(int(sha256[0],16)): {bin(int(sha256[0],16))}')  
    print(f'bin(int(sha256[0],16))[2:]: {bin(int(sha256[0],16))[2:]}')      
    chksum = bin(int(sha256[0],16))[2:].zfill(4)
    print(f'chksum: {chksum}')    
    g11 = binstr + chksum
    print(f'g11str: {g11}')
    chunks = [g11[0+i:11+i] for i in range(0, len(g11), 11)]
    print(f'chunks: {chunks}')
    indeces = [int(c,2) for c in chunks]
    print(f'indeces: {indeces}')    
    words = [wordlist[i] for i in indeces]
    return words

# test
number = 0
words = ['abandon', 'abandon', 'abandon', 'abandon', 'abandon', 'abandon', 
         'abandon', 'abandon', 'abandon', 'abandon', 'abandon', 'about']
w = number_to_words(number, wordlist)
assert w == words

# get random number
n = random.getrandbits(128)
print(n)
print(number_to_words(n, wordlist))

binstr: 00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
hexstr: 00000000000000000000000000000000
ascstr: b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
sha256: 374708fff7719dd5979ec875d56cd2286f6d3cf7ec317a3b25632aab28ec37bb
sha256[0]: 3
int(sha256[0],16): 3
bin(int(sha256[0],16)): 0b11
bin(int(sha256[0],16))[2:]: 11
chksum: 0011
g11str: 000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011
chunks: ['00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000000', '00000000011']
indeces: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]
263322929485946354894299764901556348565
binstr: 11000110000110100010011011001000010100001101001011010001100101001010100000110101110100100010111000011011110111010110011010010101
hexstr: C61A2

In [79]:
# binstr = bin(int(hex(random.getrandbits(128)),16))[2:].zfill(128)

In [80]:
def words_to_seed(words):
    password = ' '.join(words).encode('utf-8')
    seed = hashlib.pbkdf2_hmac('sha512', password, 'mnemonic'.encode('utf-8'), 2048)
    return seed

# test
seed = words_to_seed(words)
print(seed)

b'^\xb0\x0b\xbd\xdc\xf0i\x08H\x89\xa8\xab\x91UV\x81e\xf5\xc4S\xcc\xb8^p\x81\x1a\xae\xd6\xf6\xda_\xc1\x9aZ\xc4\x0b8\x9c\xd3p\xd0\x86 m\xec\x8a\xa6\xc4=\xae\xa6i\x0f \xad=\x8dH\xb2\xd2\xce\x9e8\xe4'


In [81]:
# test using mnemonic package from trezor
from mnemonic import Mnemonic
mnemo = Mnemonic("english")
seed2 = mnemo.to_seed(password, passphrase="")
print(seed2)

b'^\xb0\x0b\xbd\xdc\xf0i\x08H\x89\xa8\xab\x91UV\x81e\xf5\xc4S\xcc\xb8^p\x81\x1a\xae\xd6\xf6\xda_\xc1\x9aZ\xc4\x0b8\x9c\xd3p\xd0\x86 m\xec\x8a\xa6\xc4=\xae\xa6i\x0f \xad=\x8dH\xb2\xd2\xce\x9e8\xe4'


In [82]:
def b58encode(v: bytes) -> str:
    alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"

    p, acc = 1, 0
    for c in reversed(v):
        acc += p * c
        p = p << 8

    string = ""
    while acc:
        acc, idx = divmod(acc, 58)
        string = alphabet[idx : idx + 1] + string
    return string

In [83]:
# seed to bip44 master private key
bcseed = hmac.new(b"Bitcoin seed", seed, digestmod=hashlib.sha512).digest()
# Serialization format https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki#Serialization_format
xprv = b"\x04\x88\xad\xe4"  # private mainnet
# xprv = b"\x04\x35\x83\x94"  # private testnet
xprv += b"\x00" * 9  # Depth, parent fingerprint, and child number
xprv += bcseed[32:]  # Chain code
xprv += b"\x00" + bcseed[:32]  # Master key
hashed_xprv = hashlib.sha256(xprv).digest()
hashed_xprv = hashlib.sha256(hashed_xprv).digest()
xprv += hashed_xprv[:4]
key = b58encode(xprv)
key


'xprv9s21ZrQH143K3GJpoapnV8SFfukcVBSfeCficPSGfubmSFDxo1kuHnLisriDvSnRRuL2Qrg5ggqHKNVpxR86QEC8w35uxmGoggxtQTPvfUu'